In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import evaluate
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report

/home/rjledesma/python-projects/venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
dataset = load_dataset("ag_news")

train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

label_names = dataset["train"].features["label"].names

print(label_names)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

['World', 'Sports', 'Business', 'Sci/Tech']


In [3]:
#tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(example["text"], truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [4]:
#model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [10]:
trainer.train()

Step,Training Loss
50,1.134031
100,0.534230
150,0.432285
200,0.397821
250,0.380267
300,0.284453
350,0.226578
400,0.316189
450,0.256274
500,0.269556


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.42316833305358886, metrics={'train_runtime': 19.7427, 'train_samples_per_second': 202.607, 'train_steps_per_second': 25.326, 'total_flos': 85111269402432.0, 'train_loss': 0.42316833305358886, 'epoch': 2.0})

In [11]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Step,Accuracy
0.269556,0.429237,500,0.880000


{'eval_loss': 0.429237425327301, 'eval_accuracy': 0.88}


In [12]:
preds = trainer.predict(test_dataset)

y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.82      0.87       120
           1       0.95      0.99      0.97       121
           2       0.87      0.81      0.84       134
           3       0.80      0.90      0.85       125

    accuracy                           0.88       500
   macro avg       0.88      0.88      0.88       500
weighted avg       0.88      0.88      0.88       500



In [ ]:
#pca eval
words = [
    "government","president","election","war","country",
    "basketball","football","team","player","coach",
    "business","market","stock","money","company",
    "computer","technology","science","software","internet"
]

embedding_layer = model.get_input_embeddings()

vectors = []
valid_words = []

for word in words:
    ids = tokenizer.encode(word, add_special_tokens=False)

    if len(ids) > 0:
        tensor = torch.tensor(ids).to(model.device)

        with torch.no_grad():
            vec = embedding_layer(tensor).mean(dim=0).detach().cpu().numpy()

        vectors.append(vec)
        valid_words.append(word)

vectors = np.array(vectors)

pca = PCA(n_components=2)
reduced = pca.fit_transform(vectors)

plt.figure(figsize=(10,7))
plt.scatter(reduced[:,0], reduced[:,1])

for i, word in enumerate(valid_words):
    plt.annotate(word, (reduced[i,0], reduced[i,1]))

plt.title("PCA Visualization")
plt.show()

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.